# B3 · SQLAlchemy & GeoAlchemy2 — the database as Python objects

**What they are:**
- **SQLAlchemy** manages connections (an *engine* keeps a pool of open lines) and can describe tables and queries as Python objects.
  GeoPandas `read_postgis`/`to_postgis` use a SQLAlchemy engine.
- **GeoAlchemy2** teaches SQLAlchemy about PostGIS: a `Geometry` column type and all `ST_` functions via `func.ST_...`.

**Two styles:**
- **Core**: write SQL-like expressions in Python (`select(table).where(...)`).
- **ORM** (Object Relational Mapper): one Python **class** = one table, one **object** = one row.

**Daily picture:** Psycopg is phoning the database and speaking SQL yourself. SQLAlchemy is a switchboard with a translator:
you speak Python, it speaks SQL for you, and it keeps the lines open so the next call is quicker.

**Where this fits in your plan:** you need the *engine* for B4. The ORM matters if you later build a web app (Flask, FastAPI) on PostGIS.

⚙️ The database from **B0** must be running.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
import sqlalchemy as sa
from sqlalchemy import create_engine, text, select, func
import geoalchemy2
from geotrain.db import DSN
URL = DSN.replace("postgresql://", "postgresql+psycopg://", 1)     # tell SQLAlchemy to use psycopg 3
print("SQLAlchemy", sa.__version__, "| GeoAlchemy2", geoalchemy2.__version__)

---
## Level 1 — Basics: engine and text queries

**Goal of this level:** create an engine, run SQL text with bound parameters, and read results.

**Daily-life picture:** Getting a switchboard installed once, then making many calls through it.

### Exercise 1.1 — Create an engine

**Step 1 · Purpose.** What is the **purpose** of `create_engine(url)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`engine = create_engine(URL)`. Then `with engine.connect() as conn: conn.execute(text("SELECT 1"))`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
engine = ____(URL)
with engine.connect() as conn:
    print(conn.execute(text("SELECT postgis_version()")).scalar())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Creates an engine: it knows how to reach the database and keeps a pool of connections to reuse.

**Real life:** Created once at the start of a notebook, a script or a web app; everything else uses it.

```python
engine = create_engine(URL)
with engine.connect() as conn:
    print(conn.execute(text("SELECT postgis_version()")).scalar())
```

</details>

### Exercise 1.2 — Bound parameters in text()

**Step 1 · Purpose.** What is the **purpose** of `text('... :name').bindparams / execute(sql, {'name': value})`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`text("SELECT school FROM schools WHERE capacity > :cap")`, then pass `{"cap": 800}`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with engine.connect() as conn:
    rows = conn.execute(text("SELECT school, capacity FROM schools WHERE capacity > ____"), {"cap": ____}).all()
rows

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Placeholders written `:name`, filled safely from a dict (SQLAlchemy's version of `%s`).

**Real life:** The same safe querying as in B2, with the style used in most Python web apps.

```python
with engine.connect() as conn:
    rows = conn.execute(text("SELECT school, capacity FROM schools WHERE capacity > :cap"), {"cap": 800}).all()
rows
```

</details>

### Exercise 1.3 — Write safely: engine.begin()

**Step 1 · Purpose.** What is the **purpose** of `with engine.begin() as conn:`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Create a small table `notes (id serial, txt text)`, insert one row, then read it back with `engine.connect()`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with engine.____() as conn:
    conn.execute(text("DROP TABLE IF EXISTS notes"))
    conn.execute(text("CREATE TABLE notes (id serial PRIMARY KEY, txt text)"))
    conn.execute(text("INSERT INTO notes (txt) VALUES (:t)"), {"t": "first note"})
with engine.connect() as conn:
    print(conn.execute(text("SELECT * FROM notes")).all())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Opens a transaction that **commits automatically** at the end, or rolls back if an error happens.

**Real life:** Updating a table in one safe block: all changes or none.

```python
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS notes"))
    conn.execute(text("CREATE TABLE notes (id serial PRIMARY KEY, txt text)"))
    conn.execute(text("INSERT INTO notes (txt) VALUES (:t)"), {"t": "first note"})
with engine.connect() as conn:
    print(conn.execute(text("SELECT * FROM notes")).all())
```

</details>

---
## Level 2 — Core: tables and queries as Python objects

**Goal of this level:** describe an existing table in Python and build spatial queries with `func.ST_...`.

**Daily-life picture:** Instead of writing the letter yourself, you fill in a form and the office writes the letter.

### Exercise 2.1 — Reflect an existing table

**Step 1 · Purpose.** What is the **purpose** of `sa.Table(name, MetaData(), autoload_with=engine)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`schools = sa.Table("schools", sa.MetaData(), autoload_with=engine)`. Print each column's name and type.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
meta = sa.MetaData()
schools = sa.Table("schools", meta, autoload_with=____)
for col in schools.columns:
    print(col.name, col.type)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Reads a table's structure from the database into a Python `Table` object; GeoAlchemy2 recognises the geometry column.

**Real life:** Working with a large database whose tables you did not create, without retyping their columns.

```python
meta = sa.MetaData()
schools = sa.Table("schools", meta, autoload_with=engine)
for col in schools.columns:
    print(col.name, col.type)
```

</details>

### Exercise 2.2 — A spatial Core query

**Step 1 · Purpose.** What is the **purpose** of `select(...).where(func.ST_DWithin(...))`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Point: `func.ST_SetSRID(func.ST_Point(393000, 5821000), 32633)`. Condition: `func.ST_DWithin(schools.c.geometry, p, 1500)`. `print(stmt)` shows the SQL generated.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
p = func.ST_SetSRID(func.ST_Point(393_000, 5_821_000), 32633)
stmt = select(schools.c.school, func.ST_Distance(schools.c.geometry, p).label("dist")).where(func.____(schools.c.geometry, p, 1500))
print(stmt)
with engine.connect() as conn:
    print(conn.execute(stmt).all())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Builds a SELECT in Python; `func.ST_...` calls any PostGIS function by name.

**Real life:** Queries built step by step in code, e.g. adding filters only when a user ticks a box in an app.

```python
p = func.ST_SetSRID(func.ST_Point(393_000, 5_821_000), 32633)
stmt = select(schools.c.school, func.ST_Distance(schools.c.geometry, p).label("dist")).where(func.ST_DWithin(schools.c.geometry, p, 1500))
print(stmt)
with engine.connect() as conn:
    print(conn.execute(stmt).all())
```

</details>

---
## Level 3 — ORM: one class = one table

**Goal of this level:** define a spatial table as a Python class, add objects, and query them.

**Daily-life picture:** A class is a blank form ('Tree: species, location'); each filled-in form is one row in the table.

In [ ]:
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from geoalchemy2 import Geometry, WKTElement
from geoalchemy2.shape import to_shape, from_shape
import shapely

### Exercise 3.1 — Define a spatial class and create its table

**Step 1 · Purpose.** What is the **purpose** of `class Bench(Base): geom = mapped_column(Geometry('POINT', srid=32633))`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Type `'POINT'`, `srid=32633`. `Base.metadata.create_all(engine)` creates missing tables.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
class Base(DeclarativeBase):
    pass

class Bench(Base):
    __tablename__ = "benches"
    id: Mapped[int] = mapped_column(primary_key=True)
    condition: Mapped[str]
    geom = mapped_column(Geometry("____", srid=____))

Base.metadata.drop_all(engine)
Base.metadata.____(engine)
print(sa.inspect(engine).get_columns("benches"))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Declares a table as a Python class, including a PostGIS geometry column; `create_all` creates it in the database (with a spatial index).

**Real life:** The data model of an app where citizens report broken benches on a map.

```python
class Base(DeclarativeBase):
    pass

class Bench(Base):
    __tablename__ = "benches"
    id: Mapped[int] = mapped_column(primary_key=True)
    condition: Mapped[str]
    geom = mapped_column(Geometry("POINT", srid=32633))

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
print(sa.inspect(engine).get_columns("benches"))
```

</details>

### Exercise 3.2 — Add rows as objects

**Step 1 · Purpose.** What is the **purpose** of `session.add(Bench(..., geom=WKTElement(...)))`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`WKTElement("POINT(392800 5821400)", srid=32633)` or `from_shape(shapely.Point(...), srid=32633)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with Session(engine) as session:
    session.add(Bench(condition="broken", geom=WKTElement("POINT(392800 5821400)", srid=32633)))
    session.add(Bench(condition="ok", geom=____(shapely.Point(394_100, 5_819_800), srid=32633)))
    session.add(Bench(condition="broken", geom=from_shape(shapely.Point(395_300, 5_823_100), srid=32633)))
    session.____()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Creates row objects in Python; a `Session` collects them and writes them on `commit()`.

**Real life:** Saving a citizen's report from a web form as a new row.

```python
with Session(engine) as session:
    session.add(Bench(condition="broken", geom=WKTElement("POINT(392800 5821400)", srid=32633)))
    session.add(Bench(condition="ok", geom=from_shape(shapely.Point(394_100, 5_819_800), srid=32633)))
    session.add(Bench(condition="broken", geom=from_shape(shapely.Point(395_300, 5_823_100), srid=32633)))
    session.commit()
```

</details>

### Exercise 3.3 — Query objects and get Shapely back

**Step 1 · Purpose.** What is the **purpose** of `session.scalars(select(Bench).where(...)) + to_shape()`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`select(Bench).where(Bench.condition == "broken")`. For each bench: `to_shape(b.geom)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with Session(engine) as session:
    for b in session.scalars(select(Bench).where(Bench.condition == "____")):
        pt = ____(b.geom)
        print(b.id, b.condition, pt.x, pt.y)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Returns Bench objects matching a condition; `to_shape` turns the geometry into a Shapely object.

**Real life:** A maintenance crew's list: all broken benches and their coordinates.

```python
with Session(engine) as session:
    for b in session.scalars(select(Bench).where(Bench.condition == "broken")):
        pt = to_shape(b.geom)
        print(b.id, b.condition, pt.x, pt.y)
```

</details>

---
## Level 4 — Professional: spatial ORM queries and design choices

**Goal of this level:** combine ORM and PostGIS functions, and choose the right tool.

**Daily-life picture:** A craftsperson owns several tools and knows which one fits the job.

### Exercise 4.1 — Broken benches in the flood zone

**Scenario.** Which broken benches are within 150 m of the river? Use the ORM for benches and a reflected `river` table.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Reflect `river` like 2.1. Join: `select(Bench).join(river, func.ST_DWithin(Bench.geom, river.c.geometry, 150)).where(Bench.condition == 'broken')`.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
river = sa.Table("river", meta, autoload_with=engine)
stmt = (select(Bench.id, func.ST_Distance(Bench.geom, river.c.geometry).label("d"))
        .join(river, func.____(Bench.geom, river.c.geometry, ____))
        .where(Bench.condition == "broken"))
with Session(engine) as session:
    print(session.execute(stmt).all())

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Overlay / proximity (ORM + PostGIS functions)

Any PostGIS function can be used through `func`. The ORM builds the SQL; PostGIS (with GEOS) evaluates it.

```python
river = sa.Table("river", meta, autoload_with=engine)
stmt = (select(Bench.id, func.ST_Distance(Bench.geom, river.c.geometry).label("d"))
        .join(river, func.ST_DWithin(Bench.geom, river.c.geometry, 150))
        .where(Bench.condition == "broken"))
with Session(engine) as session:
    print(session.execute(stmt).all())
```

</details>

### Exercise 4.2 — Which tool when?

**Scenario.** Choose the best tool (Psycopg, SQLAlchemy Core, SQLAlchemy ORM + GeoAlchemy2, GeoPandas) for:
(a) a quick analysis and map of accidents per neighbourhood,
(b) a web app where users add and edit benches,
(c) a nightly bulk load of 5 million GPS points,
(d) a reusable query builder with optional filters.
Then show the SQL SQLAlchemy generates for one ORM query (`print(stmt.compile(engine))`).

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Think: who uses the result (you, an app, a pipeline)? How big is the data? Do you need objects or tables?

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
stmt = select(Bench).where(Bench.condition == "broken")
print(stmt.____(engine))

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Decision (tool choice)

(a) GeoPandas — tables and plots. (b) ORM + GeoAlchemy2 — objects, validation, sessions. (c) Psycopg COPY — speed. (d) SQLAlchemy Core — composable queries.

```python
stmt = select(Bench).where(Bench.condition == "broken")
print(stmt.compile(engine))
```

</details>

---
## 🏁 Final test

For each: **plan in words → code → interpretation.**

### Test 1 — 🛠️ Task question

**A.** With a `text()` query and bound parameters, return the neighbourhood that contains the point (394 500, 5 822 700).

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
with engine.connect() as conn:
    print(conn.execute(text("SELECT name FROM neighbourhoods WHERE ST_Contains(geometry, ST_SetSRID(ST_Point(:x, :y), 32633))"),
                       {"x": 394_500, "y": 5_822_700}).scalar())
```

</details>

### Test 2 — 🛠️ Task question

**B.** With the ORM, mark every bench within 1 km of *Central Clinic* as `"checked"`, and commit.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
clinics = sa.Table("clinics", meta, autoload_with=engine)
central = select(clinics.c.geometry).where(clinics.c.clinic == "Central Clinic").scalar_subquery()
with Session(engine) as session:
    for b in session.scalars(select(Bench).where(func.ST_DWithin(Bench.geom, central, 1000))):
        b.condition = "checked"
    session.commit()
with engine.connect() as conn:
    print(conn.execute(text("SELECT id, condition FROM benches ORDER BY id")).all())
```

</details>

### Test 3 — 🧠 Modelling question

**C · Modelling a data model.** Design the tables (classes) for a 'report a problem' app in Riverton: citizens report potholes, broken lights and fallen trees, with a photo and a status.
Which geometry type(s) and SRID? Which columns? Why?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

One table `reports`: `id`, `category` (pothole/light/tree), `status` (new/in progress/fixed), `created_at`, `photo_url`, `geom = Geometry("POINT", srid=4326)`.
- SRID **4326**, because phones send GPS lon/lat; transform to 32633 in queries when measuring (`ST_Transform`) — or store both.
- A `POINT` is enough: a citizen taps one spot.
- An index on `geom` (GIST) and on `status`.
Optional: table `users`, and a table `streets` (LINESTRING) to snap reports to the nearest street.

</details>

---
## 🏗️ Project — Backend of a 'report a problem' app with SQLAlchemy

**The brief.** Citizens will report potholes, broken lights and fallen trees in an app. Build the database side with the **ORM**:
        the table as a Python class, 60 test reports on real streets, queries the city needs, a status update, and a map in QGIS
        where staff can edit statuses.

**Your data** (made for this project, in `data/projects/`):

| Layer / file | What it holds |
|---|---|
| `projects.streets` | the street network (B0 project) |
| `public.schools` | schools (B0) |
| `projects.reports` | the table you create |

**How to work:** for each task write the plan in words, then the code, then one sentence with the result. Model answers are hidden; open them only after you tried. The last part happens in **QGIS**.

In [ ]:
from geotrain.projects import build_projects, PROJ_DIR, publish
build_projects()

import datetime
import pandas as pd
import geopandas as gpd
from sqlalchemy import text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from geoalchemy2 import Geometry
from geoalchemy2.shape import from_shape

### Project task 1

**The model.** Class `Report` → table `projects.reports`: id, category, status (default 'new'), created_at, description, `geom` POINT 32633. Create it.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
class PBase(DeclarativeBase):
    pass
class Report(PBase):
    __tablename__ = "reports"
    __table_args__ = {"schema": "projects"}
    id: Mapped[int] = mapped_column(primary_key=True)
    category: Mapped[str]
    status: Mapped[str] = mapped_column(default="new")
    created_at: Mapped[datetime.datetime]
    description: Mapped[str]
    geom = mapped_column(Geometry("POINT", srid=32633))
PBase.metadata.drop_all(engine); PBase.metadata.create_all(engine)
print(sa.inspect(engine).get_columns("reports", schema="projects")[-1]["type"])
```

</details>

### Project task 2

**Test data.** 60 reports at random points **on** streets (a random street segment, a random position along it), with random categories and times.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
import random
random.seed(4)
streets = gpd.read_postgis("SELECT street_id, kind, geometry FROM projects.streets", engine, geom_col="geometry")
cats = ["pothole", "broken light", "fallen tree"]
with Session(engine) as s:
    for i in range(60):
        seg = streets.geometry.iloc[random.randrange(len(streets))]
        p = seg.interpolate(random.random(), normalized=True)
        s.add(Report(category=random.choice(cats), description=f"test report {i}",
                     created_at=datetime.datetime(2025, 9, 1) + datetime.timedelta(hours=random.randint(0, 24 * 20)),
                     geom=from_shape(p, srid=32633)))
    s.commit()
with Session(engine) as s:
    print(s.execute(select(Report.category, func.count()).group_by(Report.category)).all())
```

</details>

### Project task 3

**Queries the city needs:** (a) open reports per category; (b) reports within 300 m of a school (reflect `public.schools`).

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
schools_t = sa.Table("schools", sa.MetaData(), autoload_with=engine, schema="public")
with Session(engine) as s:
    print(s.execute(select(Report.category, func.count()).where(Report.status == "new").group_by(Report.category)).all())
    near = s.execute(select(Report.id, Report.category, schools_t.c.school)
                     .join(schools_t, func.ST_DWithin(Report.geom, schools_t.c.geometry, 300))).all()
print(len(near), "reports near schools", near[:3])
```

</details>

### Project task 4

**Update:** main streets are repaired first, so every report on a **main** street goes to 'in progress' (a report is 'on' a street if within 15 m). Commit and count statuses.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
streets_t = sa.Table("streets", sa.MetaData(), autoload_with=engine, schema="projects")
on_main = select(Report.id).join(streets_t, func.ST_DWithin(Report.geom, streets_t.c.geometry, 15)).where(
    streets_t.c.kind == "main")
with Session(engine) as s:
    for r in s.scalars(select(Report).where(Report.id.in_(on_main))):
        r.status = "in progress"
    s.commit()
    print(s.execute(select(Report.status, func.count()).group_by(Report.status)).all())
```

</details>

### Project task 5

**Reports per tract** for the monthly report: join reports to `projects.tracts` with `ST_Within` and read the result into pandas.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
per_tract = pd.read_sql(text("""SELECT t.tract_id, count(r.id) AS reports
                                 FROM projects.tracts t LEFT JOIN projects.reports r ON ST_Within(r.geom, t.geometry)
                                 GROUP BY t.tract_id ORDER BY reports DESC"""), engine)
per_tract.head()
```

</details>

### 🗺️ Project in QGIS

1. Load **projects.reports** (it has a primary key, so QGIS can edit it). Style *Categorized* by `status`; use different symbols per `category` (*Rule-based*).
2. *Toggle Editing* → select a report → change `status` to 'fixed' in the attribute table → *Save Edits*.
3. Back in the notebook, re-run the status count of task 4: your QGIS edit is there. **QGIS and Python share one database.**
4. Try to add a new point in QGIS with the *Add Point Feature* tool: this is how an office user would log a phone report.

<details><summary>🔌 <b>How to connect QGIS to PostGIS</b> (once per computer)</summary>

1. Start the database (Docker: `docker start geotrain-db`).
2. In QGIS, open the **Browser** panel → right-click **PostgreSQL** → **New Connection…**
3. Name `geotrain`, Host `localhost`, Port `5432`, Database `geotrain`.
   Authentication → *Basic*: user `geo`, password `geo` (tick *Store*). Click **Test Connection** → OK.
4. Expand **geotrain → projects** (the schema) and **drag a table onto the map**.
5. To run SQL and see the result on the map: **Database → DB Manager → PostGIS → geotrain → SQL Window**.
   Write the query → **Execute** → tick **Load as new layer**, choose the geometry column and a unique id column → **Load**.
6. To open a GeoPackage instead: drag the `.gpkg` file from your file manager into QGIS.

</details>

### 📦 Deliverables (tick when done)

- [ ] ORM class and table `projects.reports`
- [ ] Answers to task 3 and the status table after task 4
- [ ] Reports per tract table + a QGIS edit round-trip

### 🪞 Reflection — your long-term plan

Do you expect to build an app one day, or mainly to analyse? Write which of Core / ORM / plain `text()` you will actually use, and why.

✍️ **Your answer:** …